# Qwen3-VL ChartQA — AWQ 完整品質檢查（Colab A100）

這份 notebook **只做一件事**：用 vLLM 比較 merged-16bit 與正式 AWQ W4A16 模型，在完整 ChartQA test（human／augmented 各 1,250 題）的 relaxed accuracy。

- 不會重新訓練、合併或量化模型。
- 固定使用官方 vLLM CUDA 12.9 wheel，避開 Colab 上錯裝 CUDA 13 的 `libcudart.so.13` 問題。
- 兩個模型各自在獨立子程序執行；程序結束後 GPU 記憶體會完整釋放，避免第二個模型 OOM。
- 大量 vLLM log 寫入檔案，Colab 畫面每分鐘只顯示一行心跳，避免瀏覽器卡頓。
- 每個模型完成後立刻把 predictions 上傳 HF Hub，可斷線續跑。

使用方式：選 **A100 GPU**、確認左側 Secret `HF_TOKEN` 已開放筆記本存取，然後按「全部執行」。不需要修改任何 code。


In [ ]:
# 1. 硬體檢查（安裝 vLLM 前不 import torch）
import shutil, subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True, check=True,
).stdout.strip()
print("GPU:", gpu)
print("Disk free:", round(shutil.disk_usage("/content").free / 1024**3, 1), "GB")
assert "A100" in gpu, "本流程需要 A100 40GB；請到「執行階段 → 變更執行階段類型」選 A100 後重跑。"
assert shutil.disk_usage("/content").free > 45 * 1024**3, "可用磁碟需至少 45GB。"


In [ ]:
%%capture
# 2. 固定已驗證版本；vLLM 必須在任何 import torch 之前安裝
import os, subprocess, sys
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["VLLM_LOGGING_LEVEL"] = "WARNING"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "uv"], check=True)
subprocess.run([
    "uv", "pip", "install", "--system", "--no-cache",
    "vllm==0.25.1+cu129",
    "torch==2.11.0+cu129", "torchvision==0.26.0+cu129", "torchaudio==2.11.0+cu129",
    "transformers==5.10.1", "datasets==5.0.0", "httpx", "pandas", "tabulate",
    "--extra-index-url", "https://wheels.vllm.ai/0.25.1/cu129",
    "--extra-index-url", "https://download.pytorch.org/whl/cu129",
    "--index-strategy", "unsafe-best-match",
], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "hf_xet"], check=False)


In [ ]:
# 3. HF 登入；先驗證輸入真的是 256 筆正式 AWQ，而不是 smoke 權重
import importlib.metadata, json, os, subprocess, sys
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download, login, whoami

expected_versions = {"vllm": "0.25.1+cu129", "torch": "2.11.0+cu129",
                     "torchvision": "0.26.0+cu129", "torchaudio": "2.11.0+cu129",
                     "transformers": "5.10.1", "datasets": "5.0.0"}
resolved_versions = {name: importlib.metadata.version(name) for name in expected_versions}
assert resolved_versions == expected_versions, resolved_versions
print("versions:", resolved_versions)

probe_code = ("import torch, torchvision; from vllm import LLM, SamplingParams; "
              "assert torch.version.cuda == '12.9', torch.version.cuda; "
              "assert torch.cuda.is_available(), 'CUDA unavailable'; "
              "print('vLLM CUDA import OK:', torch.__version__, torch.version.cuda)")
probe = subprocess.run([sys.executable, "-c", probe_code], capture_output=True, text=True)
print(probe.stdout.strip())
if probe.returncode != 0:
    print(probe.stderr)
assert probe.returncode == 0, "vLLM CUDA 12.9 import 失敗；請不要繼續下載模型。"

HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "找不到 Colab Secret: HF_TOKEN"
os.environ["HF_TOKEN"] = HF_TOKEN  # 讓隔離的評估子程序沿用登入，不會印出 token
login(token=HF_TOKEN, add_to_git_credential=False)
HF_USER = whoami()["name"]
api = HfApi(token=HF_TOKEN)
MERGED_REPO = f"{HF_USER}/qwen3vl-8b-chartqa-merged-16bit"
AWQ_REPO = f"{HF_USER}/qwen3vl-8b-chartqa-awq"
for repo in (MERGED_REPO, AWQ_REPO):
    assert api.repo_exists(repo), f"找不到模型 repo: {repo}"
meta_path = hf_hub_download(AWQ_REPO, "quantization_metadata.json")
config_path = hf_hub_download(AWQ_REPO, "config.json")
quant_meta = json.load(open(meta_path, encoding="utf-8"))
quant_config = json.load(open(config_path, encoding="utf-8"))["quantization_config"]
assert quant_meta["calibration_samples"] == 256, quant_meta
assert quant_meta["smoke_test"] is False, quant_meta
assert quant_meta["weight_bits"] == 4 and quant_meta["group_size"] == 32, quant_meta
group = quant_config["config_groups"]["group_0"]["weights"]
assert quant_config["quant_method"] == "compressed-tensors", quant_config
assert group["num_bits"] == 4 and group["group_size"] == 32, group
assert group["strategy"] == "group" and group["symmetric"] is True, group
assert "lm_head" in quant_config["ignore"] and any("visual" in x for x in quant_config["ignore"]), quant_config["ignore"]
print("HF user:", HF_USER)
print("formal AWQ verified: 256 samples, W4A16 g32,", quant_meta["gpu"])


In [ ]:
%%writefile /content/eval_one_model.py
# 4. 每個模型在獨立程序內執行；stdout 全寫 log，避免 Colab 前端渲染大量文字
from __future__ import annotations

import argparse, base64, importlib.metadata, io, json, os, platform, time
from pathlib import Path

os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("VLLM_LOGGING_LEVEL", "WARNING")
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

from datasets import Dataset
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from vllm import LLM, SamplingParams

DATASET_ID = "HuggingFaceM4/ChartQA"
ANSWER_INSTRUCTION = "Answer the question using a single word or phrase."
HUMAN, MACHINE = 0, 1

def retry(label, fn, attempts=8):
    last = None
    for attempt in range(1, attempts + 1):
        try:
            return fn()
        except Exception as exc:
            last = exc
            if attempt == attempts:
                raise
            wait = min(15 * attempt, 90)
            print(f"[{label}] retry {attempt}/{attempts} after {type(exc).__name__}; sleep {wait}s", flush=True)
            time.sleep(wait)
    raise last

def load_chartqa(n, human_or_machine, dataset_revision, seed=3407):
    files = sorted(
        f for f in HfApi().list_repo_files(DATASET_ID, repo_type="dataset", revision=dataset_revision)
        if f.startswith("data/test-")
    )
    assert files, "ChartQA test parquet not found"
    paths = [retry(f, lambda f=f: hf_hub_download(DATASET_ID, f, repo_type="dataset", revision=dataset_revision)) for f in files]
    ds = Dataset.from_parquet(paths)
    ds = ds.filter(lambda ex: ex["human_or_machine"] == human_or_machine)
    return ds.shuffle(seed=seed).select(range(min(n, len(ds))))

def get_answer(ex):
    label = ex["label"]
    return str(label[0]) if isinstance(label, list) else str(label)

def to_data_uri(image, max_side=1024):
    image = image.convert("RGB")
    image.thumbnail((max_side, max_side))
    buf = io.BytesIO()
    image.save(buf, format="JPEG", quality=90)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()

def message(ex):
    return [{"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": to_data_uri(ex["image"])}},
        {"type": "text", "text": f"{ex['query']}\n{ANSWER_INSTRUCTION}"},
    ]}]

def to_float(text):
    try:
        return float(text.rstrip("%")) / 100.0 if text.endswith("%") else float(text)
    except ValueError:
        return None

def normalize(text):
    text = text.strip()
    return text[:-1].rstrip() if text.endswith(".") else text

def correct(pred, target, tolerance=0.05):
    pred, target = normalize(pred), target.strip()
    pf, tf = to_float(pred), to_float(target)
    if pf is not None and tf:
        return abs(pf - tf) / abs(tf) <= tolerance
    return pred.lower() == target.lower()

def pkg(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--repo", required=True)
    parser.add_argument("--revision", required=True)
    parser.add_argument("--fingerprint", required=True)
    parser.add_argument("--dataset-revision", required=True)
    parser.add_argument("--eval-recipe", required=True)
    parser.add_argument("--tag", required=True)
    parser.add_argument("--n", type=int, default=1250)
    parser.add_argument("--output", required=True)
    args = parser.parse_args()

    print(f"prefetch model: {args.repo}", flush=True)
    model_path = retry(args.repo, lambda: snapshot_download(args.repo, revision=args.revision), attempts=10)
    splits = [("human", HUMAN), ("augmented", MACHINE)]
    datasets = {name: load_chartqa(args.n, hom, args.dataset_revision) for name, hom in splits}
    print(f"load vLLM: {args.tag}; prompts={sum(len(ds) for ds in datasets.values())}", flush=True)
    llm = LLM(
        model=model_path, max_model_len=4096, max_num_seqs=8,
        gpu_memory_utilization=0.88, enforce_eager=True, disable_log_stats=True,
        limit_mm_per_prompt={"image": 1, "video": 0},
    )
    sampling = SamplingParams(temperature=0, max_tokens=32)

    result = {}
    for name, _ in splits:
        ds = datasets[name]
        print(f"{args.tag} {name}: rendering/inference n={len(ds)}", flush=True)
        outputs = llm.chat([message(ex) for ex in ds], sampling_params=sampling, use_tqdm=True)
        preds = [out.outputs[0].text.strip() for out in outputs]
        golds = [get_answer(ex) for ex in ds]
        acc = sum(correct(p, g) for p, g in zip(preds, golds)) / len(ds)
        result[name] = {"n": len(ds), "relaxed_accuracy": acc, "predictions": preds, "golds": golds}
        print(f"{args.tag} {name}: {acc:.4f} (n={len(ds)})", flush=True)
    result["runtime"] = {
        "model": args.repo, "model_revision": args.revision,
        "model_fingerprint": args.fingerprint,
        "dataset": DATASET_ID, "dataset_revision": args.dataset_revision,
        "eval_recipe": args.eval_recipe,
        "engine": "vllm", "python": platform.python_version(),
        "packages": {name: pkg(name) for name in ["vllm", "torch", "transformers", "compressed-tensors"]},
        "max_model_len": 4096, "max_num_seqs": 8, "enforce_eager": True,
    }
    Path(args.output).parent.mkdir(parents=True, exist_ok=True)
    with open(args.output, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=1, ensure_ascii=False)
    print(f"saved: {args.output}", flush=True)

if __name__ == "__main__":
    main()


In [ ]:
# 5. 依序評估兩個模型；每分鐘只印一行狀態，完成一個就立即 push
import hashlib, json, os, shutil, signal, subprocess, sys, time
from pathlib import Path
from huggingface_hub import hf_hub_download

EVAL_N = 1250
EVAL_RECIPE = "chartqa-test-seed3407-jpeg1024q90-shortanswer-relaxed5-v1-vllm0251cu129"
DATASET_REVISION = api.dataset_info("HuggingFaceM4/ChartQA").sha
JOB_TIMEOUT_SECONDS = 3 * 60 * 60
OUT_DIR = Path("/content/eval_quant")
OUT_DIR.mkdir(exist_ok=True)
jobs = [("merged16", MERGED_REPO), ("awq", AWQ_REPO)]

def tail(path, lines=30):
    text = Path(path).read_text(encoding="utf-8", errors="replace") if Path(path).exists() else ""
    return "\n".join(text.splitlines()[-lines:])

def model_identity(repo):
    info = api.model_info(repo, files_metadata=True)
    parts = []
    for item in info.siblings:
        name = item.rfilename
        if "/" in name or not name.endswith((".json", ".jinja", ".model", ".safetensors", ".txt")):
            continue
        if name == "quantization_metadata.json":
            continue
        digest = getattr(item.lfs, "sha256", None) if item.lfs else item.blob_id
        parts.append(f"{name}:{digest}")
    assert any(".safetensors:" in part for part in parts), f"找不到模型權重: {repo}"
    fingerprint = hashlib.sha256("\n".join(sorted(parts)).encode()).hexdigest()
    return info.sha, fingerprint

def result_matches(path, repo, fingerprint):
    try:
        data = json.load(open(path, encoding="utf-8"))
        return (data.get("runtime", {}).get("model") == repo
                and data.get("runtime", {}).get("model_fingerprint") == fingerprint
                and data.get("runtime", {}).get("dataset_revision") == DATASET_REVISION
                and data.get("runtime", {}).get("eval_recipe") == EVAL_RECIPE
                and all(data.get(split, {}).get("n") == EVAL_N for split in ("human", "augmented")))
    except Exception:
        return False

def upload_with_retry(path, remote_path, attempts=6):
    for attempt in range(1, attempts + 1):
        try:
            api.upload_file(path_or_fileobj=str(path), path_in_repo=remote_path, repo_id=AWQ_REPO)
            return
        except Exception as exc:
            if attempt == attempts:
                raise
            wait = min(15 * attempt, 60)
            print(f"upload retry {attempt}/{attempts}: {type(exc).__name__}; {wait}s")
            time.sleep(wait)

def terminate_process_group(pgid):
    try:
        os.killpg(pgid, signal.SIGTERM)
    except ProcessLookupError:
        return
    time.sleep(2)
    try:
        os.killpg(pgid, signal.SIGKILL)
    except ProcessLookupError:
        pass

for tag, repo in jobs:
    revision, fingerprint = model_identity(repo)
    assert revision, f"無法取得模型 revision: {repo}"
    filename = f"preds_{tag}_n{EVAL_N}.json"
    output = OUT_DIR / filename
    remote_path = f"eval_quant/{filename}"
    if output.exists() and result_matches(output, repo, fingerprint):
        upload_with_retry(output, remote_path)
        print(f"[{tag}] 本機已有完整結果，已補上傳至 HF Hub。")
        continue
    try:
        cached = hf_hub_download(AWQ_REPO, remote_path)
        shutil.copy2(cached, output)
        if result_matches(output, repo, fingerprint):
            print(f"[{tag}] HF Hub 已有相同模型權重的正式 n={EVAL_N} 結果，直接重用。")
            continue
        print(f"[{tag}] Hub 結果不符合目前模型權重，安全重算。")
        output.unlink(missing_ok=True)
    except Exception:
        pass

    output.unlink(missing_ok=True)
    log_path = OUT_DIR / f"{tag}.log"
    cmd = [sys.executable, "/content/eval_one_model.py",
           "--repo", repo, "--revision", revision, "--fingerprint", fingerprint,
           "--dataset-revision", DATASET_REVISION, "--eval-recipe", EVAL_RECIPE,
           "--tag", tag,
           "--n", str(EVAL_N), "--output", str(output)]
    env = os.environ.copy()
    env.update({"PYTHONUNBUFFERED": "1", "VLLM_LOGGING_LEVEL": "WARNING",
                "VLLM_WORKER_MULTIPROC_METHOD": "spawn", "HF_HUB_DISABLE_XET": "1",
                "TOKENIZERS_PARALLELISM": "false"})
    print(f"[{tag}] 開始；詳細 log -> {log_path}")
    started = time.time()
    proc = None
    try:
        with open(log_path, "w", encoding="utf-8") as log_file:
            proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT,
                                    env=env, start_new_session=True)
            while True:
                try:
                    proc.wait(timeout=60)
                    break
                except subprocess.TimeoutExpired:
                    elapsed = time.time() - started
                    mb = log_path.stat().st_size / 1024**2 if log_path.exists() else 0
                    print(f"[{tag}] still running: {elapsed/60:.0f} min | log {mb:.1f} MB")
                    if elapsed > JOB_TIMEOUT_SECONDS:
                        raise TimeoutError(f"{tag} 超過 3 小時安全上限")
    except BaseException:
        if proc is not None:
            terminate_process_group(proc.pid)
        print(tail(log_path, 80))
        raise
    terminate_process_group(proc.pid)  # 清掉 vLLM 可能殘留的 EngineCore 子程序
    if proc.returncode != 0:
        print(tail(log_path, 80))
        raise RuntimeError(f"{tag} 評估失敗（exit={proc.returncode}）；請依上方 log 尾段排查。")
    assert output.exists(), f"缺少輸出: {output}"
    print(tail(log_path, 20))
    upload_with_retry(output, remote_path)
    print(f"[{tag}] 完成並已 push。子程序已退出，GPU 記憶體已完整釋放。")
    time.sleep(5)


In [ ]:
# 6. 產生量化前後表格、品質門檻與 summary，push 到 HF Hub
import json
import pandas as pd

res16 = json.load(open(OUT_DIR / f"preds_merged16_n{EVAL_N}.json", encoding="utf-8"))
res4 = json.load(open(OUT_DIR / f"preds_awq_n{EVAL_N}.json", encoding="utf-8"))
rows = []
for split in ["human", "augmented"]:
    assert res16[split]["n"] == res4[split]["n"] == EVAL_N
    a16 = res16[split]["relaxed_accuracy"]
    a4 = res4[split]["relaxed_accuracy"]
    rows.append({"test split": split, "n": EVAL_N,
                 "merged-16bit": round(a16, 4), "awq-w4a16-g32": round(a4, 4),
                 "Δ": round(a4 - a16, 4)})
n_total = 2 * EVAL_N
overall16 = sum(res16[s]["relaxed_accuracy"] * EVAL_N for s in ["human", "augmented"]) / n_total
overall4 = sum(res4[s]["relaxed_accuracy"] * EVAL_N for s in ["human", "augmented"]) / n_total
rows.append({"test split": "overall", "n": n_total,
             "merged-16bit": round(overall16, 4), "awq-w4a16-g32": round(overall4, 4),
             "Δ": round(overall4 - overall16, 4)})
df = pd.DataFrame(rows)
print(df.to_markdown(index=False))
drop_pp = (overall16 - overall4) * 100
print(f"AWQ accuracy change: {(overall4-overall16)*100:+.2f} pp")
if drop_pp > 2.0:
    print(f"WARNING: AWQ 掉分 {drop_pp:.2f}pp > 2pp 品質門檻，暫停 benchmark 並重新檢視量化設定。")
else:
    print("PASS: AWQ 掉分未超過 2pp，可進入 serving benchmark。")
summary = {
    "merged": MERGED_REPO, "awq": AWQ_REPO, "eval_n_per_split": EVAL_N,
    "metric": "relaxed_accuracy(5%)", "engine": "vllm isolated subprocess",
    "quality_gate_max_drop_pp": 2.0, "quality_gate_passed": drop_pp <= 2.0,
    "table": rows, "runtime": {"merged": res16.get("runtime"), "awq": res4.get("runtime")},
}
summary_path = OUT_DIR / "results.json"
json.dump(summary, open(summary_path, "w", encoding="utf-8"), indent=1, ensure_ascii=False)
upload_with_retry(summary_path, "eval_quant/results.json")
print(f"pushed -> https://huggingface.co/{AWQ_REPO}/tree/main/eval_quant")


## 完成後

最後一格的表格即為品質門檻結果。若顯示 `PASS`，下一步才執行獨立的 vLLM benchmark notebook。跑完請「中斷連線並刪除執行階段」。
